# Paired-book wait-cost diagnostic

## tl;dr

Across the preregistered 120–179 second interval, `1,403 / 1,440` condition-seconds had valid pairs. The other `37` seconds came from two episodes in one market, and both outcome books were invalid in every one of them. Therefore the paired-book requirement created `0 / 2,880` pair-only orientation-second exposures: it never delayed an otherwise individually valid chosen side, so no recovery wait or executable-ask erosion was observable. Together with the independently observed zero residual rejections, this active mechanism made no structural decision difference in this capture. The rule remains frozen pending the sealed forward score; no parameter, threshold, production, or active-block change follows from this label-free result.


## Context & Methods

The active rule requires both outcome books to be valid before selecting a baseline-eligible row. Prior diagnostics showed that the registered residual threshold never rejected a valid pair, making paired-book availability the only observed source of selectivity. This notebook addresses the remaining operational question: if one side is valid but its complement is not, how long until a valid pair returns and how much does the already-valid side's executable ask move?

### Key Assumptions

- The cohort is the preregistered 120–179 second candidate interval in 24 non-overlapping BTC five-minute markets from the retained July 15 diagnostic capture.
- End-of-second states preserve native event order and clamp each condition's source timestamp to its monotone seen maximum, matching the prior structural diagnostic.
- A pair-only exposure exists only when the chosen orientation is individually valid and the opposite side is invalid. Rows where the chosen side itself is invalid are not attributed to the pair gate.
- Recovery is the first later end-of-second state in the same market where both books are valid and the frozen residual rule passes.
- Ask increase is edge erosion dollar-for-dollar before latency, traversal, fills, and fees. This is structural feasibility evidence, not prediction or profitability evidence.


In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import math
import statistics
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
CAPTURE = Path('/private/tmp/fresh-block-canary-recovered/segment_001')
CONVERTED = CAPTURE / 'converted_v10'
MANIFEST_PATH = CONVERTED / 'manifest.json'
OUTPUT_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_paired_book_wait_cost_diagnostic.json'
PRIOR_DIAGNOSTIC_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_independence_diagnostic.json'

MAX_BOOK_AGE_SECONDS = 30.0
DEPTH_LEVELS = 3
EPSILON = 1e-12

manifest = json.loads(MANIFEST_PATH.read_text())
prior_diagnostic = json.loads(PRIOR_DIAGNOSTIC_PATH.read_text())
assert manifest['stats']['skipped_malformed_raw'] == 0
assert manifest['stats']['skipped_unknown_market'] == 0
assert manifest['stats']['skipped_unknown_token'] == 0

event_files = [Path(row['path']) for row in manifest['hours']]
assert all(path.exists() for path in event_files)
markets = manifest['markets']
assert len(markets) == 24

def parse_utc(value: str) -> float:
    return datetime.fromisoformat(value.replace('Z', '+00:00')).timestamp()

market_windows = {}
tokens_by_condition = defaultdict(dict)
for condition_id, market in markets.items():
    close_ts = parse_utc(market['end_date'])
    market_windows[condition_id] = {'open_ts': close_ts - 300.0, 'close_ts': close_ts}
    for outcome in market['outcomes']:
        name = outcome['name'].lower()
        assert name in {'up', 'down'}
        tokens_by_condition[condition_id][name] = outcome['token_id']
assert all(set(pair) == {'up', 'down'} for pair in tokens_by_condition.values())

print({
    'conditions': len(markets),
    'event_files': [path.name for path in event_files],
    'resolution_manifest_loaded': False,
    'terminal_labels_loaded': False,
    'strategy_outcomes_loaded': False,
    'active_forward_block_loaded': False,
})


{'conditions': 24, 'event_files': ['2026-07-15T06.v1.candles.jsonl.gz', '2026-07-15T07.v1.candles.jsonl.gz', '2026-07-15T08.v1.candles.jsonl.gz'], 'resolution_manifest_loaded': False, 'terminal_labels_loaded': False, 'strategy_outcomes_loaded': False, 'active_forward_block_loaded': False}


In [2]:
def fresh_book_state() -> dict:
    return {
        'bids': {}, 'asks': {}, 'best_bid': 0.0, 'best_ask': 0.0,
        'last_update': None, 'has_snapshot': False,
    }

def quantile(values: list[float], probability: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight

def distribution(values: list[float]) -> dict:
    return {
        'count': len(values),
        'min': min(values) if values else None,
        'p50': quantile(values, 0.50),
        'p90': quantile(values, 0.90),
        'p99': quantile(values, 0.99),
        'max': max(values) if values else None,
        'mean': statistics.fmean(values) if values else None,
    }

def apply_event(book: dict, event: dict, effective_ts: float) -> None:
    if event['ev'] == 'book':
        book['bids'] = {float(price): float(size) for price, size in event['bids'] if float(size) > 0}
        book['asks'] = {float(price): float(size) for price, size in event['asks'] if float(size) > 0}
        book['has_snapshot'] = True
    elif event['ev'] == 'chg':
        side = 'bids' if event['s'] == 'BUY' else 'asks'
        price = float(event['p'])
        size = float(event['sz'])
        if size > 0:
            book[side][price] = size
        else:
            book[side].pop(price, None)
    else:
        raise AssertionError(f'unexpected event type {event["ev"]}')
    book['best_bid'] = float(event['bb'])
    book['best_ask'] = float(event['ba'])
    book['last_update'] = effective_ts

def book_metrics(book: dict, cutoff: float) -> tuple[dict | None, str | None]:
    if not book['has_snapshot'] or book['last_update'] is None:
        return None, 'missing_snapshot'
    age_s = cutoff - book['last_update']
    if age_s < -1e-9:
        return None, 'future_update'
    if age_s > MAX_BOOK_AGE_SECONDS:
        return None, 'stale_book'
    best_bid = book['best_bid']
    best_ask = book['best_ask']
    if not (0.0 < best_bid < best_ask < 1.0):
        return None, 'invalid_top'
    bid_levels = sorted(
        ((price, size) for price, size in book['bids'].items() if size > 0 and price <= best_bid + 1e-9),
        reverse=True,
    )[:DEPTH_LEVELS]
    ask_levels = sorted(
        ((price, size) for price, size in book['asks'].items() if size > 0 and price >= best_ask - 1e-9)
    )[:DEPTH_LEVELS]
    bid_depth = sum(size for _, size in bid_levels)
    ask_depth = sum(size for _, size in ask_levels)
    if bid_depth <= 0 or ask_depth <= 0:
        return None, 'missing_positive_depth'
    midpoint = (best_bid + best_ask) / 2.0
    microprice = (best_ask * bid_depth + best_bid * ask_depth) / (bid_depth + ask_depth)
    return {
        'best_bid': best_bid,
        'best_ask': best_ask,
        'midpoint': midpoint,
        'microprice': microprice,
        'age_s': age_s,
    }, None


In [3]:
books = defaultdict(fresh_book_state)
condition_tick = defaultdict(lambda: 0.01)
last_raw_ts = defaultdict(lambda: float('-inf'))
last_effective_ts = defaultdict(lambda: float('-inf'))
next_second = {condition_id: int(window['open_ts']) for condition_id, window in market_windows.items()}
native_timestamp_regressions = Counter()
event_counts = Counter()
samples = []

def capture_second(condition_id: str, second: int) -> None:
    window = market_windows[condition_id]
    elapsed_s = second - int(window['open_ts'])
    cutoff = second + 1.0 - 1e-9
    pair = tokens_by_condition[condition_id]
    up, up_reason = book_metrics(books[pair['up']], cutoff)
    down, down_reason = book_metrics(books[pair['down']], cutoff)
    pair_valid = up is not None and down is not None
    fixed_pass = False
    if pair_valid:
        max_residual = max(
            abs(up['midpoint'] + down['midpoint'] - 1.0),
            abs(up['microprice'] + down['microprice'] - 1.0),
        )
        fixed_pass = max_residual <= 2.0 * condition_tick[condition_id] + EPSILON
    samples.append({
        'condition_id': condition_id,
        'second': second,
        'elapsed_s': elapsed_s,
        'candidate_interval': 120 <= elapsed_s < 180,
        'up': up,
        'down': down,
        'up_reason': up_reason,
        'down_reason': down_reason,
        'pair_valid': pair_valid,
        'fixed_pass': fixed_pass,
    })

for event_path in event_files:
    with gzip.open(event_path, 'rt') as handle:
        for line in handle:
            event = json.loads(line)
            if event['ev'] == 'trade':
                event_counts['trade'] += 1
                continue
            condition_id = event['mkt']
            if condition_id not in markets:
                continue
            raw_ts = float(event['ts'])
            if raw_ts + 1e-9 < last_raw_ts[condition_id]:
                native_timestamp_regressions[condition_id] += 1
            last_raw_ts[condition_id] = max(last_raw_ts[condition_id], raw_ts)
            effective_ts = max(last_effective_ts[condition_id], raw_ts)
            last_effective_ts[condition_id] = effective_ts
            close_second = int(market_windows[condition_id]['close_ts'])
            while next_second[condition_id] < close_second and next_second[condition_id] + 1.0 - 1e-9 < effective_ts:
                capture_second(condition_id, next_second[condition_id])
                next_second[condition_id] += 1
            apply_event(books[event['tok']], event, effective_ts)
            event_counts[event['ev']] += 1
            top_prices = [float(event['bb']), float(event['ba'])]
            if any(price > 0 and (price < 0.04 or price > 0.96) for price in top_prices):
                condition_tick[condition_id] = 0.001

for condition_id, window in market_windows.items():
    close_second = int(window['close_ts'])
    while next_second[condition_id] < close_second:
        capture_second(condition_id, next_second[condition_id])
        next_second[condition_id] += 1

assert len(samples) == 24 * 300
assert sum(row['candidate_interval'] for row in samples) == 24 * 60
print({
    'events': dict(event_counts),
    'samples': len(samples),
    'candidate_interval_samples': sum(row['candidate_interval'] for row in samples),
    'native_timestamp_regressions': sum(native_timestamp_regressions.values()),
})


{'events': {'book': 120475, 'chg': 7561554}, 'samples': 7200, 'candidate_interval_samples': 1440, 'native_timestamp_regressions': 865}


In [4]:
samples_by_condition = defaultdict(list)
for row in samples:
    samples_by_condition[row['condition_id']].append(row)
for rows in samples_by_condition.values():
    rows.sort(key=lambda row: row['second'])

candidate_rows = [row for row in samples if row['candidate_interval']]
invalid_candidate_rows = [row for row in candidate_rows if not row['pair_valid']]
exposures = []

for row in invalid_candidate_rows:
    condition_rows = samples_by_condition[row['condition_id']]
    for direction, opposite_direction in [('up', 'down'), ('down', 'up')]:
        chosen = row[direction]
        opposite = row[opposite_direction]
        if chosen is None or opposite is not None:
            continue
        recovery = next(
            (
                future for future in condition_rows
                if future['second'] > row['second'] and future['pair_valid'] and future['fixed_pass']
            ),
            None,
        )
        if recovery is None:
            exposures.append({
                'condition_id': row['condition_id'],
                'second': row['second'],
                'elapsed_s': row['elapsed_s'],
                'direction': direction,
                'opposite_invalid_reason': row[f'{opposite_direction}_reason'],
                'recovered': False,
            })
            continue
        ask_change = recovery[direction]['best_ask'] - chosen['best_ask']
        exposures.append({
            'condition_id': row['condition_id'],
            'second': row['second'],
            'elapsed_s': row['elapsed_s'],
            'direction': direction,
            'opposite_invalid_reason': row[f'{opposite_direction}_reason'],
            'recovered': True,
            'recovery_delay_seconds': recovery['second'] - row['second'],
            'ask_before': chosen['best_ask'],
            'ask_after': recovery[direction]['best_ask'],
            'ask_change': ask_change,
            'adverse_ask_change': max(ask_change, 0.0),
        })

recoveries = [row for row in exposures if row['recovered']]
unrecovered = [row for row in exposures if not row['recovered']]

episodes = []
for condition_id, rows in samples_by_condition.items():
    interval = [row for row in rows if row['candidate_interval']]
    index = 0
    while index < len(interval):
        if interval[index]['pair_valid']:
            index += 1
            continue
        start = index
        while index + 1 < len(interval) and not interval[index + 1]['pair_valid']:
            index += 1
        run = interval[start:index + 1]
        episodes.append({
            'condition_id': condition_id,
            'start_elapsed_s': run[0]['elapsed_s'],
            'end_elapsed_s': run[-1]['elapsed_s'],
            'duration_sample_seconds': len(run),
            'up_valid_seconds': sum(row['up'] is not None for row in run),
            'down_valid_seconds': sum(row['down'] is not None for row in run),
        })
        index += 1

invalid_reason_counts = Counter()
for row in invalid_candidate_rows:
    for reason in (row['up_reason'], row['down_reason']):
        if reason:
            invalid_reason_counts[reason] += 1

results = {
    'candidate_interval': {
        'condition_seconds': len(candidate_rows),
        'orientation_seconds': 2 * len(candidate_rows),
        'valid_pair_condition_seconds': sum(row['pair_valid'] for row in candidate_rows),
        'invalid_pair_condition_seconds': len(invalid_candidate_rows),
        'valid_pair_coverage': sum(row['pair_valid'] for row in candidate_rows) / len(candidate_rows),
        'conditions_with_pair_invalidity': len({row['condition_id'] for row in invalid_candidate_rows}),
        'invalid_reason_counts_by_side': dict(invalid_reason_counts),
    },
    'pair_gate_exposure': {
        'pair_only_exposed_orientation_seconds': len(exposures),
        'share_of_all_orientation_seconds': len(exposures) / (2 * len(candidate_rows)),
        'conditions': len({row['condition_id'] for row in exposures}),
        'directions': dict(Counter(row['direction'] for row in exposures)),
        'recovered_within_market': len(recoveries),
        'unrecovered_within_market': len(unrecovered),
        'recovery_delay_seconds': distribution([row['recovery_delay_seconds'] for row in recoveries]),
        'chosen_ask_change': distribution([row['ask_change'] for row in recoveries]),
        'adverse_chosen_ask_change': distribution([row['adverse_ask_change'] for row in recoveries]),
        'ask_deteriorations': sum(row['ask_change'] > EPSILON for row in recoveries),
        'ask_unchanged': sum(abs(row['ask_change']) <= EPSILON for row in recoveries),
        'ask_improvements': sum(row['ask_change'] < -EPSILON for row in recoveries),
        'opposite_invalid_reasons': dict(Counter(row['opposite_invalid_reason'] for row in exposures)),
    },
    'invalid_pair_episodes': {
        'count': len(episodes),
        'duration_sample_seconds': distribution([row['duration_sample_seconds'] for row in episodes]),
        'conditions': len({row['condition_id'] for row in episodes}),
    },
}

assert results['candidate_interval']['invalid_pair_condition_seconds'] == 37
assert results['candidate_interval']['valid_pair_condition_seconds'] == 1403
assert results['pair_gate_exposure']['pair_only_exposed_orientation_seconds'] <= 37

print(json.dumps(results, indent=2, sort_keys=True))


{
  "candidate_interval": {
    "condition_seconds": 1440,
    "conditions_with_pair_invalidity": 1,
    "invalid_pair_condition_seconds": 37,
    "invalid_reason_counts_by_side": {
      "invalid_top": 74
    },
    "orientation_seconds": 2880,
    "valid_pair_condition_seconds": 1403,
    "valid_pair_coverage": 0.9743055555555555
  },
  "invalid_pair_episodes": {
    "conditions": 1,
    "count": 2,
    "duration_sample_seconds": {
      "count": 2,
      "max": 27,
      "mean": 18.5,
      "min": 10,
      "p50": 18.5,
      "p90": 25.3,
      "p99": 26.830000000000002
    }
  },
  "pair_gate_exposure": {
    "adverse_chosen_ask_change": {
      "count": 0,
      "max": null,
      "mean": null,
      "min": null,
      "p50": null,
      "p90": null,
      "p99": null
    },
    "ask_deteriorations": 0,
    "ask_improvements": 0,
    "ask_unchanged": 0,
    "chosen_ask_change": {
      "count": 0,
      "max": null,
      "mean": null,
      "min": null,
      "p50": null,
      "

In [5]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {path.name: sha256_file(path) for path in event_files}
source_hashes[MANIFEST_PATH.name] = sha256_file(MANIFEST_PATH)
prior_hashes = prior_diagnostic['source_authority']['sha256']
for path in event_files:
    assert source_hashes[path.name] == prior_hashes[path.name]
assert source_hashes[MANIFEST_PATH.name] == prior_hashes[MANIFEST_PATH.name]

evidence = {
    'schema_version': 1,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'mechanism_id': 'binary_complement_coherence_v1',
    'status': 'LABEL_FREE_PAIRED_BOOK_WAIT_COST_QUANTIFIED_ACTIVE_RULE_UNCHANGED',
    'decision_question': 'When paired-book validity blocks an otherwise individually valid orientation, how long does recovery take and how much does the executable ask move?',
    'source_authority': {
        'capture': 'abandoned fresh-block-canary captured 2026-07-15T06:49Z through 2026-07-15T08:50Z',
        'capture_conditions': len(markets),
        'distilled_files': [path.name for path in event_files],
        'sha256': source_hashes,
        'resolution_manifest_loaded': False,
        'terminal_labels_loaded': False,
        'strategy_outcomes_loaded': False,
        'active_forward_block_loaded': False,
    },
    'data_quality': {
        'grain': 'one causal end-of-second paired-book state per condition-second, expanded to two orientations only for pair-gate exposure',
        'native_timestamp_regressions_observed': sum(native_timestamp_regressions.values()),
        'timestamp_policy': 'preserve native file order; clamp each condition source timestamp to its monotone seen maximum before sampling',
        'skipped_malformed_raw': manifest['stats']['skipped_malformed_raw'],
        'skipped_unknown_market': manifest['stats']['skipped_unknown_market'],
        'skipped_unknown_token': manifest['stats']['skipped_unknown_token'],
        'promotion_or_exact_replay_eligible': False,
        'quality_assessment': 'SHARE_WITH_CAVEATS_FOR_LABEL_FREE_OPERATIONAL_DIAGNOSTIC_ONLY',
    },
    'methodology': {
        'candidate_interval_elapsed_seconds': [120, 180],
        'sampling': 'end-of-second causal state',
        'pair_only_exposure': 'chosen side individually valid while the opposite side is invalid',
        'recovery': 'first later same-condition sample where both books are valid and the frozen two-tick residual rule passes',
        'ask_change': 'recovery chosen ask minus pre-delay chosen ask; positive values erode edge',
        'alternate_thresholds_tested': 0,
    },
    'structural_results': results,
    'mechanism_assessment': {
        'residual_clause_observed_role': 'NO_INCREMENTAL_REJECTIONS_IN_SOURCE_CAPTURE',
        'paired_validity_observed_role': 'ONLY_SOURCE_OF_POTENTIAL_SELECTION_DELAY_IN_SOURCE_CAPTURE',
        'active_binary_complement_rule_changed': False,
        'interpretation': 'This diagnostic bounds how often paired validity can delay an otherwise book-valid orientation and the associated top-ask movement. It cannot determine whether those orientation-seconds are actual baseline candidates or whether a delay removes losses.',
    },
    'decision': {
        'strategy_adjustment': 'NO_PARAMETER_OR_RULE_CHANGE_DURING_FROZEN_FORWARD_BLOCK',
        'research_action': 'Use schema-6 post-floor timing and decision-ask attribution as authoritative; do not infer loss removal or profitability from this label-free feasibility bound.',
        'a_plus_claim': False,
        'profitability_claim': False,
        'live_trading': 'OFF',
    },
    'limitations': [
        'Only 24 diagnostic markets from one capture are observed.',
        'Potential orientations are not strategy candidates because signal, edge, state, and terminal outcomes are intentionally absent.',
        'One-hertz sampling can miss sub-second invalidity and recovery.',
        'Ask change omits measured latency, depth traversal, fills, fees, and any direction change while waiting.',
        'Timestamp clamping prevents time travel but cannot reconstruct an unavailable independent receive timestamp.',
    ],
}

temporary = OUTPUT_PATH.with_name(f'{OUTPUT_PATH.name}.tmp')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(OUTPUT_PATH)

print({
    'artifact': str(OUTPUT_PATH.relative_to(ROOT)),
    'pair_only_exposures': len(exposures),
    'max_recovery_delay_seconds': results['pair_gate_exposure']['recovery_delay_seconds']['max'],
    'max_adverse_ask_change': results['pair_gate_exposure']['adverse_chosen_ask_change']['max'],
    'status': evidence['status'],
})


{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_paired_book_wait_cost_diagnostic.json', 'pair_only_exposures': 0, 'max_recovery_delay_seconds': None, 'max_adverse_ask_change': None, 'status': 'LABEL_FREE_PAIRED_BOOK_WAIT_COST_QUANTIFIED_ACTIVE_RULE_UNCHANGED'}


## Results

- Valid-pair coverage was `1,403 / 1,440` (`97.43%`) in the registered candidate interval.
- The `37` invalid condition-seconds formed two episodes of `10` and `27` seconds in one market.
- Both books were invalid in all `37` seconds, yielding `0 / 2,880` pair-only orientation-second exposures.
- Because no otherwise-valid chosen orientation was blocked, observable recovery delay and chosen-ask erosion counts were both zero.

## Takeaways

- Paired-book validity added no observed decision selectivity beyond the chosen-book validity already required by the baseline in this capture.
- The frozen residual clause had already added zero observed rejections, so neither component demonstrated structural selectivity here.
- This is a strong negative feasibility result but not a profitability result: actual strategy candidates, labels, outcomes, and the active forward block were not loaded.
- Schema-6 post-floor attribution remains authoritative for actual decision disagreements, winner retention, loss removal, delay, and fee-aware economics. No new hypothesis or rule change is justified before that sealed score.
